In [59]:
!pip install transformers datasets huggingface_hub transformers[torch] accelerate --upgrade

In [60]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

In [61]:
from huggingface_hub import login

login()

In [62]:
import re
from sklearn.model_selection import train_test_split

In [63]:
f = open("./drive/MyDrive/metriccoders_datasets/history_of_india.txt", "r")
text = f.readlines()

In [64]:
print(len(text))

1325


In [65]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

Train dataset length: 1126
Test dataset length: 199


In [66]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [67]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [68]:
from transformers import TextDataset, DataCollatorForLanguageModeling
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [69]:
def load_dataset(train_path, test_path, tokeinzer):
  train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=64)
  test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=64)
  data_collator = DataCollatorForLanguageModeling(
          tokenizer=tokenizer, mlm=False,
  )
  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [70]:
training_args = TrainingArguments(
    output_dir="./gpt2-history-of-india",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_steps=400,
    save_steps=100,
    save_total_limit=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [71]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=38, training_loss=3.7146176789936267, metrics={'train_runtime': 23.6348, 'train_samples_per_second': 51.196, 'train_steps_per_second': 1.608, 'total_flos': 39520419840000.0, 'train_loss': 3.7146176789936267, 'epoch': 2.0})

In [72]:
trainer.save_model()

In [73]:
input_text = "India is the"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

In [74]:
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

In [75]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

India is the world's largest producer of rice, with a population of over 1.5 million. The country is also the world's largest producer of iron ore, which is used in the manufacture of many products such as cooking oil, rice, meat, and vegetables. The country is also the world's largest producer of iron ore, which is used in the manufacture of many products such as cooking oil, meat, and vegetables. India is the world's largest producer of coal, which is used in


In [81]:
from huggingface_hub import notebook_login, create_repo, Repository
notebook_login()


In [83]:
repo_name = "fine-tuned-gpt2-history-of-india"  # Change this to your desired repository name
from huggingface_hub import HfApi

# Initialize the HfApi instance
api = HfApi()

# Create a new repository
username = api.whoami()['name']  # Get your Hugging Face username
full_repo_name = f"{username}/{repo_name}"

# Create the repository (you can also create it on the Hugging Face website)
api.create_repo(repo_name, private=False)

api.upload_folder(
    folder_path='./gpt2-history-of-india',  # Path to the folder with your model
    repo_id=full_repo_name,  # Model repository name
    commit_message="GPT-2 India History"
)

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

optimizer.pt:   0%|          | 0.00/996M [00:00<?, ?B/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Upload 10 LFS files:   0%|          | 0/10 [00:00<?, ?it/s]

...

events.out.tfevents.1724321366.c7e210015443.927.0:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

events.out.tfevents.1724321528.c7e210015443.927.1:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

events.out.tfevents.1724321735.c7e210015443.927.2:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/metriccoders/fine-tuned-gpt2-history-of-india/commit/11a3ca76f745716cee8f67498bc9c470033e7910', commit_message='GPT-2 India History', commit_description='', oid='11a3ca76f745716cee8f67498bc9c470033e7910', pr_url=None, pr_revision=None, pr_num=None)